| [0.0 Data Collection](00_data_collection.ipynb) | [1.0 Preprocessing](01_data_preprocessing.ipynb) | [2.0 Spatial Autocorrelation](02_spatial_autocorrelation.ipynb) | 3.0 MGWR | [4.0 ML Classification](04_ml_classification.ipynb) |

# 3.0 | Multi-scale Geographically Weighted Regression (MGWR)

This notebook investigates **spatially varying relationships** between accident rates and  
socio-economic/infrastructure variables. The analysis follows a structured progression:

1. **VIF check** — assess multicollinearity before regression
2. **OLS baseline** — global regression + residual Moran's I to justify spatial models
3. **MGWR fitting** — adaptive kernel with AICc bandwidth selection
4. **Coefficient surface maps** — visualise how relationships vary across space

**Dependent variable**: `log(accident_rate_per_10k)`  
**Independent variables**: `imd_score`, `junction_density`, `road_density`, `urban_pct`, `dark_pct`, `wet_road_pct`

**Warning**: MGWR fitting takes approximately 2-4 hours. Results are cached after the first run.

In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import (
    MSOA_ANALYSIS_GPKG, MGWR_COEFFICIENTS_GPKG,
    FIGURES_DIR, FIGURE_DPI
)

plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3
})

In [ ]:
gdf = gpd.read_file(MSOA_ANALYSIS_GPKG)
gdf = gdf[gdf['accident_rate_per_10k'] > 0].copy()

print(f"Loaded {len(gdf):,} MSOAs")
print(f"CRS: {gdf.crs}")
print(f"\nVariables for regression:")
from src.analysis.mgwr_analysis import INDEPENDENT_VARS
for var in INDEPENDENT_VARS:
    if var in gdf.columns:
        print(f"  {var:25s} mean={gdf[var].mean():.3f}  std={gdf[var].std():.3f}")

## 3.1 | VIF Multicollinearity Check

The Variance Inflation Factor (VIF) measures how much the variance of a regression coefficient  
is inflated due to multicollinearity. A common rule of thumb:

| VIF | Interpretation |
|-----|----------------|
| 1 | No collinearity |
| 1-5 | Moderate — acceptable |
| 5-10 | High — may need attention |
| >10 | Severe — consider removing variable |

In [ ]:
from src.analysis.mgwr_analysis import check_vif

vif_df = check_vif(gdf)

print("VIF Results")
print("═" * 40)
for _, row in vif_df.iterrows():
    flag = '  ⚠️' if row['VIF'] > 5 else '  ✓'
    print(f"  {row['variable']:25s} VIF = {row['VIF']:6.2f}{flag}")
print("═" * 40)

if (vif_df['VIF'] > 10).any():
    print("\n⚠️ WARNING: Some variables have VIF > 10. Consider removing them.")
else:
    print("\n→ All VIF values are within acceptable range.")

vif_df

## 3.2 | OLS Baseline Regression

A global OLS regression provides a baseline for comparison with spatially varying models.  
Critically, **residual Moran's I** tests whether the OLS residuals exhibit spatial autocorrelation —  
if significant, this justifies the need for GWR/MGWR.

In [ ]:
from src.analysis.mgwr_analysis import run_ols_baseline

ols = run_ols_baseline(gdf)

print("OLS Regression Results")
print("═" * 60)
print(f"  R²:       {ols['r_squared']:.4f}")
print(f"  Adj R²:   {ols['adj_r_squared']:.4f}")
print(f"  AIC:      {ols['aic']:.1f}")
print(f"")
print(f"  Residual Moran's I: {ols['residual_moran_I']:.4f}")
print(f"  Residual Moran p:   {ols['residual_moran_p']:.6f}")
print("═" * 60)

print(f"\nCoefficients:")
print(f"{'Variable':>25s}  {'Coeff':>10s}  {'Sig.':>5s}")
print("─" * 45)
for var, coef in ols['coefficients'].items():
    pval = ols['p_values'][var]
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f"  {var:>25s}  {coef:>10.4f}  {sig:>5s}")

if ols['residual_moran_p'] < 0.05:
    print(f"\n→ Significant spatial autocorrelation in OLS residuals (p = {ols['residual_moran_p']:.6f}).")
    print("  This violates the independence assumption and justifies GWR/MGWR.")

## 3.3 | MGWR Fitting

Multi-scale Geographically Weighted Regression (MGWR) extends GWR by allowing **each variable  
to operate at its own spatial scale**. Variables with large bandwidths have near-global effects,  
while those with small bandwidths exhibit localised relationships.

**Note**: First run takes approximately 2-4 hours. Subsequent runs load cached results.

In [ ]:
from src.analysis.mgwr_analysis import run_mgwr

mgwr_results = run_mgwr(gdf, cache=True)

print("Model Comparison")
print("═" * 50)
print(f"{'Model':>8s}  {'R²':>8s}  {'AICc':>12s}")
print("─" * 50)
print(f"{'OLS':>8s}  {ols['r_squared']:>8.4f}  {ols['aic']:>12.1f}")
print(f"{'GWR':>8s}  {mgwr_results['gwr_r2']:>8.4f}  {mgwr_results['gwr_aicc']:>12.1f}")
print(f"{'MGWR':>8s}  {mgwr_results['mgwr_r2']:>8.4f}  {mgwr_results['mgwr_aicc']:>12.1f}")
print("═" * 50)

### 3.3 | MGWR Bandwidths

The bandwidth for each variable indicates the spatial scale at which that variable's effect operates.  
Smaller bandwidths = more localised effects; larger bandwidths = more global effects.

In [ ]:
print("MGWR Variable Bandwidths")
print("═" * 50)
print(f"{'Variable':>25s}  {'Bandwidth':>12s}  {'Scale':>10s}")
print("─" * 50)

n_obs = len(gdf)
var_names = mgwr_results['variable_names']
bws = ['global'] + list(mgwr_results['mgwr_bw'])

for var, bw in zip(var_names, bws):
    if isinstance(bw, (int, float)):
        scale = 'Local' if bw < n_obs * 0.3 else 'Regional' if bw < n_obs * 0.7 else 'Global'
        print(f"  {var:>25s}  {bw:>12.0f}  {scale:>10s}")
    else:
        print(f"  {var:>25s}  {str(bw):>12s}  {'Global':>10s}")
print("═" * 50)

### 3.3 | Coefficient surfaces

Summary statistics of the local MGWR coefficients across all MSOAs.

In [ ]:
coef_gdf = mgwr_results['coef_gdf']
print(f"Coefficient GeoDataFrame shape: {coef_gdf.shape}")
print(f"\nCoefficient summary statistics:")
coef_gdf[INDEPENDENT_VARS].describe()

### 3.3 | Figure 6 | MGWR coefficient surface maps

These maps show how the relationship between each independent variable and accident rate  
varies spatially. Red indicates a positive association (higher values of the variable are  
associated with higher accident rates); blue indicates a negative association.

In [ ]:
from src.visualization.maps import plot_mgwr_coefficients
plot_mgwr_coefficients(coef_gdf, variables=INDEPENDENT_VARS)

In [ ]:
# Inline display
n_vars = len(INDEPENDENT_VARS)
n_cols = 3
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 7 * n_rows))
axes = axes.flatten()

for i, var in enumerate(INDEPENDENT_VARS):
    ax = axes[i]
    vmax = max(abs(coef_gdf[var].quantile(0.02)), abs(coef_gdf[var].quantile(0.98)))
    coef_gdf.plot(column=var, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  legend=True, ax=ax, edgecolor='face', linewidth=0.1,
                  legend_kwds={'shrink': 0.6})
    ax.set_title(var.replace('_', ' ').title(), fontweight='bold', fontsize=11)
    ax.axis('off')

for j in range(n_vars, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('MGWR Local Coefficients', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 3.4 | Model Comparison Summary

Comparing OLS, GWR, and MGWR on fit statistics. Lower AICc and higher R² indicate better fit.

In [ ]:
comparison = pd.DataFrame({
    'Model': ['OLS', 'GWR', 'MGWR'],
    'R²': [ols['r_squared'], mgwr_results['gwr_r2'], mgwr_results['mgwr_r2']],
    'AICc': [ols['aic'], mgwr_results['gwr_aicc'], mgwr_results['mgwr_aicc']],
})

print("\nModel Fit Comparison")
print("═" * 35)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#95a5a6', '#3498db', '#e74c3c']

axes[0].bar(comparison['Model'], comparison['R²'], color=colors, edgecolor='white')
for i, v in enumerate(comparison['R²']):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
axes[0].set_ylabel('R²')
axes[0].set_title('Goodness of Fit (R²)', fontweight='bold')
axes[0].set_ylim(0, 1)

axes[1].bar(comparison['Model'], comparison['AICc'], color=colors, edgecolor='white')
for i, v in enumerate(comparison['AICc']):
    axes[1].text(i, v + 50, f'{v:.0f}', ha='center', fontweight='bold')
axes[1].set_ylabel('AICc')
axes[1].set_title('Model Parsimony (AICc — lower is better)', fontweight='bold')

plt.tight_layout()
plt.show()

## 3.5 | Summary

**Key findings from MGWR analysis:**

1. **OLS residuals** exhibit significant spatial autocorrelation, confirming that global  
   regression inadequately captures spatial heterogeneity in accident determinants.

2. **MGWR outperforms** both OLS and GWR in terms of R² and AICc, demonstrating that  
   different variables operate at different spatial scales.

3. **Variable bandwidths** reveal that some factors (e.g., IMD deprivation) have relatively  
   global effects, while others (e.g., road infrastructure) vary more locally.

4. **Coefficient maps** show substantial spatial heterogeneity — the same variable can have  
   opposite effects in different regions of England.

Proceed to **Notebook 04** for machine learning classification.

---
| [0.0 Data Collection](00_data_collection.ipynb) | [1.0 Preprocessing](01_data_preprocessing.ipynb) | [2.0 Spatial Autocorrelation](02_spatial_autocorrelation.ipynb) | 3.0 MGWR | [4.0 ML Classification](04_ml_classification.ipynb) |